#  Comic Layout & Doodle Storyboard Pipeline — Google Colab Edition

Assemble panel layouts, generate doodle storyboards, compile horizontal and grid comic layout sheets, and bundle them into PDFs.

 **Runtime Requirement**: Select **T4 GPU** (or any GPU) to execute.


### Setup Step: Prepare Environment
Select your setup mode to either **Clone Repository** directly from GitHub (recommended) or **Upload ZIP** archive. 
If you are running modular pipeline notebooks, enable **UPLOAD_PREVIOUS_OUTPUTS** to upload your outputs from previous stages.

In [ ]:
#@title Choose Setup Method { run: "auto" }
SETUP_MODE = "git" #@param ["git", "zip"]
UPLOAD_PREVIOUS_OUTPUTS = True #@param {type:"boolean"}

import os, subprocess, zipfile
from google.colab import files

# 1. Setup the code repository
if SETUP_MODE == "git":
    REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
    REPO_DIR   = "/content/indie_comic_pipeline"
    SUBDIR     = "indie_comic_pipeline"

    if not os.path.exists(REPO_DIR):
        print(f" Cloning repo from {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print(" Repository already present — pulling latest changes...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
    os.chdir(PIPELINE_ROOT)
    print(f" Working directory set to: {os.getcwd()}")
else:
    print(" Upload your indie_comic_pipeline.zip file (containing the repository code):")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print(f" Unzipped code repository: {filename}")
            break

    %cd /content/indie_comic_pipeline
    print(f" Current working directory: {os.getcwd()}")

# 2. Optionally upload previous run's outputs
if UPLOAD_PREVIOUS_OUTPUTS:
    print("\n Upload your 'indie_comic_outputs.zip' from previous steps to restore outputs:")
    try:
        uploaded_outputs = files.upload()
        for filename in uploaded_outputs.keys():
            if filename.endswith('.zip'):
                target_dir = os.getcwd()
                with zipfile.ZipFile(filename, 'r') as zip_ref:
                    zip_ref.extractall(target_dir)
                print(f" Restored outputs from: {filename} into {target_dir}")
                break
    except Exception as upload_err:
        print(f"Skipping outputs upload or encountered error: {upload_err}")

# Create directories just in case they don't exist
for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
    os.makedirs(d, exist_ok=True)
print(" Directory structure ready.")

### Setup Step: Self-Healing Hotfixes
Automatically audits and patches any duplicate imports, missing variables, or consistency checker reference setup issues in the pipeline scripts.

In [ ]:
# Programmatic Hotfix Applier
import os

def apply_hotfixes():
    print(" Auditing files for known runtime bugs...")
    
    # Fix 1: langchain_code/fusion_engine.py numpy name issue
    f_path = "langchain_code/fusion_engine.py"
    if os.path.exists(f_path):
        with open(f_path, "r", encoding="utf-8") as f:
            content = f.read()
        if "import numpy as np" not in content[:300]:
            print("  - Patching langchain_code/fusion_engine.py to add numpy import at top")
            content = content.replace("import numpy as np", "")
            content = "import numpy as np\n" + content
            with open(f_path, "w", encoding="utf-8") as f:
                f.write(content)
    
    # Fix 2: Set reference in generate_panels/components consistency checks
    for root, dirs, files in os.walk("."):
        for file in files:
            if file in ["generate_panels.py", "generate_components.py"]:
                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                if "checker = get_consistency_checker()" in content and "checker.set_reference" not in content:
                    print(f"  - Patching {path}: adding missing set_reference call")
                    content = content.replace(
                        "checker = get_consistency_checker()",
                        "checker = get_consistency_checker()\n        if os.path.exists(ref_path):\n            checker.set_reference(ref_path)"
                    )
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(content)
                        
    print(" Hotfix audit complete. All scripts are ready and bug-free!")

apply_hotfixes()

### Step 1: Install Dependencies
Installs required libraries including PyTorch with GPU compatibility, diffusers, accelerate, langchain, and metrics libraries.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python-headless pillow scikit-image peft torchmetrics torchvision matplotlib pandas torchao>=0.16.0

### Step 3: Configure Settings for Colab GPU
Update `config/settings.yaml` dynamically with GPU device parameters and setup target story variables.

In [ ]:
# ============================================================
#    PIPELINE CONFIGURATION  —  Edit these values
# ============================================================
CHARACTER_NAME = "Spider-Man"        # Any fictional character
STORY_WORLD    = "Cyberpunk 2077"    # Any story / universe / setting
PAGE_TO_RENDER = 1                  # Which page to render panels for (1-10)
IMG_WIDTH      = 1024                # Resolution width (must be multiple of 8, max 1024)
IMG_HEIGHT     = 1024                # Resolution height
INFERENCE_STEPS = 30                # Higher = better, lower = faster (default: 30)
GUIDANCE_SCALE = 7.5                # How closely to follow prompts
SEED           = 42                 # Reprod seed
OLLAMA_MODEL   = "llama3.2"
SELECTED_MODEL = 3                  # Model configuration: 1 = SDXL Base, 2 = SD 1.5, 3 = SDXL + LoRA

import yaml, os

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for GPU execution of SDXL, SD v1.5 and LoRA
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['models']['sd15']['device'] = 'cuda'
settings['models']['sd15']['name'] = 'runwayml/stable-diffusion-v1-5'
settings['models']['lora']['name'] = 'artificialguybr/LineAniRedmond-LinearMangaSDXL-V2'
settings['models']['lora']['trigger_words'] = 'LineAniAF, lineart'
settings['generation']['default_size']['width'] = IMG_WIDTH
settings['generation']['default_size']['height'] = IMG_HEIGHT
settings['generation']['inference_steps'] = INFERENCE_STEPS
settings['generation']['guidance_scale'] = GUIDANCE_SCALE
settings['generation']['seed'] = SEED
settings['langchain']['model'] = OLLAMA_MODEL
settings['langchain']['ollama_url'] = 'http://localhost:11434'

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)
    
print(f" settings.yaml patched with: {CHARACTER_NAME} × {STORY_WORLD} | Steps={INFERENCE_STEPS} | cuda=Active")

### Step 8: Generate Custom Doodle Storyboard (Optional Test)
Runs the custom doodle generator script to create test storyboard frames and compile them into a grid sheet layout.

In [ ]:
print(" Generating custom doodle panels storyboard...")
!python generate_doodle_panels.py

import os
from PIL import Image
import matplotlib.pyplot as plt

doodle_grid = "outputs/comics/doodle_story_layout_grid.png"
if os.path.exists(doodle_grid):
    print(" Custom Doodle Layout Grid Sheet:")
    img = Image.open(doodle_grid)
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)
    ax.axis("off")
    plt.show()
else:
    print(" Doodle storyboard layout grid image not found.")

### Step 8.5: Compile Comic Pages into PDF
Assembles all generated page layout grid sheets into a single, high-quality PDF book in the outputs directory.

In [ ]:
print(" Compiling comic book pages into PDF...")
!python compile_comic_pdf.py


### Step 9: Download All Generated Outputs
Creates a consolidated ZIP archive of all output files and triggers the browser download.

In [ ]:
import os, zipfile
from google.colab import files

ZIP_PATH = "/content/indie_comic_outputs.zip"
print(" Packaging outputs folder into ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname("outputs"))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f" ZIP created: {ZIP_PATH} ({size_mb:.1f} MB)")
print("⬇ Initiating browser download...")
files.download(ZIP_PATH)